# 🎯 小升初数学 · 测验系统

本系统支持：
- **按知识点测验**：专项突破薄弱环节
- **综合测验**：随机抽题，模拟真实考试
- **错题回顾**：自动记录错题，针对性复习
- **学情分析**：按知识点统计正确率，定位薄弱环节

---

In [16]:
# === 初始化 ===
import sys, os, json, random
from datetime import datetime
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
from knowledge_base import *
from IPython.display import display, Markdown, HTML, clear_output

# 学习记录文件
RECORD_FILE = 'data/study_record.json'

def load_records():
    """加载学习记录"""
    if os.path.exists(RECORD_FILE):
        with open(RECORD_FILE, 'r', encoding='utf-8') as f:
            return json.load(f)
    return {'history': [], 'wrong_questions': []}

def save_records(records):
    """保存学习记录"""
    os.makedirs(os.path.dirname(RECORD_FILE), exist_ok=True)
    with open(RECORD_FILE, 'w', encoding='utf-8') as f:
        json.dump(records, f, ensure_ascii=False, indent=2)

records = load_records()
print('✅ 测验系统已就绪！')
print(f'📊 历史记录：已完成 {len(records["history"])} 次测验')

✅ 测验系统已就绪！
📊 历史记录：已完成 0 次测验


## 📋 选择测验模式

运行下方代码查看可选的知识点：

In [17]:
# 列出所有有练习题的知识点
available_topics = {}
idx = 1
md = '| 编号 | 知识点 | 题目数量 |\n'
md += '|:----:|--------|:--------:|\n'
md += f'| **0** | **🎲 综合随机测验（所有知识点）** | **{sum(len(v) for v in EXERCISES.values())}** |\n'

for topic, exs in EXERCISES.items():
    md += f'| {idx} | {topic} | {len(exs)} |\n'
    available_topics[idx] = topic
    idx += 1

display(Markdown(md))
print(f'\n输入 0 进行综合测验，或输入编号进行专项测验。')

| 编号 | 知识点 | 题目数量 |
|:----:|--------|:--------:|
| **0** | **🎲 综合随机测验（所有知识点）** | **510** |
| 1 | 数的分类 | 63 |
| 2 | 分数的意义、分类与性质 | 5 |
| 3 | 负数 | 6 |
| 4 | 因数与倍数 | 14 |
| 5 | 2、3、5倍数的特征 | 9 |
| 6 | 质数与合数 | 7 |
| 7 | 乘法分配律 | 5 |
| 8 | 解方程 | 14 |
| 9 | 百分数的概念 | 6 |
| 10 | 比和比例的基本概念 | 22 |
| 11 | 长方形与正方形 | 43 |
| 12 | 三角形 | 31 |
| 13 | 圆 | 15 |
| 14 | 长方体与正方体 | 27 |
| 15 | 圆柱 | 10 |
| 16 | 圆锥 | 8 |
| 17 | 平均数 | 9 |
| 18 | 行程问题基础（路程=速度×时间） | 35 |
| 19 | 工程问题基础题型 | 25 |
| 20 | 水费的分段计费问题 | 10 |
| 21 | 长度、面积、体积单位换算 | 20 |
| 22 | 相遇问题 | 8 |
| 23 | 增长与减少百分之几 | 8 |
| 24 | 正比例和反比例 | 6 |
| 25 | 简便计算综合 | 10 |
| 26 | 数的组成（大数读写改写） | 14 |
| 27 | 利率税收问题 | 6 |
| 28 | 折扣问题 | 11 |
| 29 | 用分数表示数量和占比 | 14 |
| 30 | 浓度问题 | 10 |
| 31 | 事件的确定性与不确定性 | 8 |
| 32 | 利用比例尺解决问题 | 2 |
| 33 | 可能性的大小 | 1 |
| 34 | 用字母表示数 | 4 |
| 35 | 条形统计图 | 4 |
| 36 | 按比分配问题 | 2 |
| 37 | 读图与图象分析 | 5 |
| 38 | 加法与乘法运算定律（交换律、结合律） | 7 |
| 39 | 含字母的式子比较大小 | 2 |
| 40 | 折线统计图 | 2 |
| 41 | 扇形统计图 | 2 |



输入 0 进行综合测验，或输入编号进行专项测验。


## 🚀 开始测验

修改 `TEST_MODE` 选择测验模式，然后运行：

In [ ]:
# ✏️ 设置测验参数
TEST_MODE = 0      # 0=综合随机, 或输入上表编号进行专项测验
NUM_QUESTIONS = 5   # 每次测验的题目数量

# === 统一样式 ===
QUIZ_STYLE = """
<style>
.quiz-banner {
    background: linear-gradient(90deg, #e53e3e 0%, #c53030 100%);
    color: white; padding: 14px 24px; border-radius: 10px;
    margin: 8px 0 16px 0; font-size: 20px; font-weight: bold;
    display: flex; justify-content: space-between; align-items: center;
}
.quiz-banner .qb-sub { font-size: 14px; opacity: 0.9; }
.q-card {
    border: 1px solid #e2e8f0; border-radius: 10px;
    padding: 18px 22px; margin: 14px 0; background: #fff;
    box-shadow: 0 1px 4px rgba(0,0,0,0.06);
}
.q-card:hover { box-shadow: 0 3px 10px rgba(0,0,0,0.12); }
.q-header {
    display: flex; justify-content: space-between; align-items: center;
    margin-bottom: 12px; padding-bottom: 10px; border-bottom: 1px dashed #e2e8f0;
}
.q-num { font-weight: bold; font-size: 16px; color: #2d3748; }
.q-topic { font-size: 12px; color: #718096; background: #edf2f7;
    padding: 2px 10px; border-radius: 10px; }
.q-tags { display: flex; gap: 6px; align-items: center; }
.qtag { padding: 2px 10px; border-radius: 12px; font-size: 12px;
    font-weight: 600; color: white; }
.qtag-choice { background: #4299e1; }
.qtag-fill { background: #48bb78; }
.qtag-judge { background: #ed8936; }
.qtag-calc { background: #9f7aea; }
.qtag-answer { background: #e53e3e; }
.qtag-sort { background: #38b2ac; }
.qtag-other { background: #a0aec0; }
.qdiff-1 { background: #c6f6d5; color: #276749; }
.qdiff-2 { background: #fefcbf; color: #975a16; }
.qdiff-3 { background: #fed7d7; color: #9b2c2c; }
.q-text { font-size: 15px; line-height: 1.9; color: #1a202c; margin: 8px 0; }
.q-opts { list-style: none; padding: 0; margin: 10px 0 0 8px; }
.q-opts li { padding: 6px 12px; margin: 4px 0; background: #f7fafc;
    border-radius: 6px; border: 1px solid #edf2f7; font-size: 14px; line-height: 1.6; }
.q-tip { background: #ebf8ff; border: 1px solid #90cdf4; border-radius: 8px;
    padding: 10px 16px; margin: 8px 0 16px 0; color: #2a69ac; font-size: 14px; }
.grade-correct { border-left: 4px solid #48bb78; background: linear-gradient(135deg, #f0fff4 0%, #fff 100%); }
.grade-wrong { border-left: 4px solid #e53e3e; background: linear-gradient(135deg, #fff5f5 0%, #fff 100%); }
.grade-icon { font-size: 22px; }
.grade-answer { font-weight: bold; margin: 4px 0; }
.grade-explain { font-size: 14px; color: #4a5568; background: #f7fafc;
    padding: 8px 12px; border-radius: 6px; margin-top: 8px; line-height: 1.8; }
.score-card {
    background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
    color: white; border-radius: 12px; padding: 24px 30px; margin: 20px 0;
    text-align: center; box-shadow: 0 4px 12px rgba(102,126,234,0.3);
}
.score-num { font-size: 48px; font-weight: bold; }
.score-label { font-size: 14px; opacity: 0.9; margin-top: 4px; }
.score-comment { font-size: 16px; margin-top: 12px; }
.wrong-card {
    border: 1px solid #fed7d7; border-left: 4px solid #e53e3e;
    border-radius: 0 10px 10px 0; padding: 16px 22px; margin: 12px 0;
    background: #fff; box-shadow: 0 1px 4px rgba(0,0,0,0.06);
}
.wrong-header { font-weight: bold; color: #9b2c2c; margin-bottom: 8px; }
.wrong-detail { font-size: 14px; line-height: 1.8; color: #4a5568; }
</style>
"""

# === 题型标签 ===
_QTAG = {'选择':'qtag-choice','填空':'qtag-fill','判断':'qtag-judge',
         '计算':'qtag-calc','解答':'qtag-answer','排序':'qtag-sort'}
_QLBL = {'选择':'选择题','填空':'填空题','判断':'判断题',
         '计算':'计算题','解答':'解答题','排序':'排序题'}
_DLBL = {1:'基础', 2:'提高', 3:'挑战'}

def _qcard(n, q):
    """生成测验题目 HTML 卡片"""
    et = q.get('type','解答')
    tc = _QTAG.get(et,'qtag-other')
    tl = _QLBL.get(et, et+'题')
    d = q.get('difficulty',1)
    opts = ''
    if 'options' in q and q['options']:
        items = ''.join(f'<li>{o}</li>' for o in q['options'])
        opts = f'<ul class="q-opts">{items}</ul>'
    qt = q.get('question','').replace('\n','<br>')
    return f"""
    <div class="q-card">
      <div class="q-header">
        <div><span class="q-num">第 {n} 题</span>
          <span class="q-topic">{q.get('_topic','')}</span></div>
        <div class="q-tags">
          <span class="qtag {tc}">{tl}</span>
          <span class="qtag qdiff-{d}">{_DLBL.get(d,'基础')}</span>
        </div>
      </div>
      <div class="q-text">{qt}</div>{opts}
    </div>"""

# === 组卷 ===
if TEST_MODE == 0:
    all_exercises = []
    for topic, exs in EXERCISES.items():
        for ex in exs:
            all_exercises.append({**ex, '_topic': topic})
    test_title = '综合随机测验'
else:
    topic_name = available_topics.get(TEST_MODE)
    if not topic_name:
        print(f'编号 {TEST_MODE} 不存在')
        all_exercises = []
    else:
        all_exercises = [{**ex, '_topic': topic_name} for ex in EXERCISES[topic_name]]
        test_title = f'专项测验：{topic_name}'

if all_exercises:
    quiz_questions = random.sample(all_exercises, min(NUM_QUESTIONS, len(all_exercises)))

    display(HTML(QUIZ_STYLE))
    display(HTML(f"""
    <div class="quiz-banner">
      <span>🎯 {test_title}</span>
      <span class="qb-sub">共 {len(quiz_questions)} 题</span>
    </div>
    <div class="q-tip">📝 认真审题，在下方单元格填写答案后运行批改。</div>"""))

    cards = ''.join(_qcard(i, q) for i, q in enumerate(quiz_questions, 1))
    display(HTML(cards))

# 🎯 综合随机测验

共 **5** 题，请在每题下方写下你的答案。

---

### 第 1 题 ⭐　`[圆]`

**【解答题】** 如图，厚度为0.25旁米的铜版纸被卷成一个空心圆柱（纸卷得很紧，没有空隙），它的外直径是180厘米，内直径是50厘米。这卷铜版纸的总长是多少米？
(8分）

...

**【解答题】** 已知如图，三角形ABC的面积为8平方座米，AE=D，BD=≤BC阴影部分的面积。

### 第 5 题 ⭐⭐　`[长度、面积、体积单位换算]`

**【解答题】** 一巩工租，甲、乙合作要12.天完成，若甲先做3关后，再由乙工作8天，共完成这项工作的若这件工作由乙独做完需要几天？

20我国是永资题比贫乏的国家之一，为加公民节水效识，合理利用水资，某市规定的居民用水收费标准如下：每户每月用水量不超过6立方米时（.合含6立方米），水费按“基本单价”收：超过6立方米时，超出的部分按“超山单价收穿。下面是张老师家3月份、4月份的用水量及水费情况。如果他家5月份用水盘是8立方米；请你算算孙老师家5月份的水是多少元？
月份用水盘（立方米）
水费（元）

---

✍️ 做完所有题目后，在下方单元格填写答案并运行批改！


## ✍️ 提交答案

在 `my_answers` 列表中填写你的答案（按题目顺序），然后运行批改：

In [ ]:
# ✏️ 在这里填写你的答案（按顺序填写，字符串格式）
# 例如：['A', '72, 4', 'x = 5', '3570', '对']
# 答案列表会根据题目数量自动生成

if 'quiz_questions' not in dir() or not quiz_questions:
    print('请先运行上方代码生成试题！')
else:
    if 'my_answers' not in dir() or len(my_answers) != len(quiz_questions) or all(a == '' for a in my_answers):
        my_answers = [''] * len(quiz_questions)
        print(f'请在 my_answers 列表中填写 {len(quiz_questions)} 个答案：')
        for i in range(len(quiz_questions)):
            print(f'   my_answers[{i}] = ""  # 第{i+1}题')
        print('\n填写完毕后，再次运行此单元格进行批改。')
    else:
        # === 自动批改 ===
        correct_count = 0
        total = len(quiz_questions)
        wrong_list = []

        grade_cards = ''
        for i, q in enumerate(quiz_questions):
            student_ans = my_answers[i].strip() if i < len(my_answers) else ''
            correct_ans = str(q['answer']).strip()
            is_correct = student_ans.replace(' ', '').lower() == correct_ans.replace(' ', '').lower()

            if is_correct:
                correct_count += 1
                cls = 'grade-correct'
                icon = '✅'
                expl_html = ''
            else:
                cls = 'grade-wrong'
                icon = '❌'
                wrong_list.append(q)
                exp = str(q.get('explanation', '待补充')).replace('\n', '<br>')
                expl_html = f'<div class="grade-explain"><b>解析：</b>{exp}</div>'

            grade_cards += f"""
            <div class="q-card {cls}">
              <div class="q-header">
                <div><span class="grade-icon">{icon}</span>
                  <span class="q-num"> 第 {i+1} 题</span>
                  <span class="q-topic">{q.get('_topic','')}</span></div>
              </div>
              <div class="q-text" style="font-size:14px; color:#4a5568;">{q.get('question','')[:120]}</div>
              <div class="grade-answer">你的答案：<span style="color:#4299e1">{student_ans if student_ans else '（未作答）'}</span></div>
              <div class="grade-answer">正确答案：<span style="color:#38a169">{correct_ans}</span></div>
              {expl_html}
            </div>"""

        score = round(correct_count / total * 100)
        if score == 100:
            comment = '完美！全部正确！你已经掌握了这些知识点！'
            emoji = '🏆'
        elif score >= 80:
            comment = '很棒！大部分都对了，再看看错题就更好了！'
            emoji = '👍'
        elif score >= 60:
            comment = '继续努力！建议复习一下出错的知识点。'
            emoji = '💪'
        else:
            comment = '需要加油哦！建议回到教学系统重新学习相关知识点。'
            emoji = '📖'

        # 薄弱知识点
        weak_html = ''
        if wrong_list:
            wrong_topics = list(set(q['_topic'] for q in wrong_list))
            topics_str = '、'.join(wrong_topics)
            weak_html = f'<div style="margin-top:12px;font-size:14px;opacity:0.9;">薄弱知识点：{topics_str}</div>'

        display(HTML(QUIZ_STYLE))
        display(HTML(f"""
        <div class="score-card">
          <div class="score-num">{score}</div>
          <div class="score-label">正确 {correct_count} / {total} 题</div>
          <div class="score-comment">{emoji} {comment}</div>
          {weak_html}
        </div>"""))
        display(HTML(grade_cards))

        # 保存记录
        _title = test_title if 'test_title' in dir() else '测验'
        test_record = {
            'date': datetime.now().strftime('%Y-%m-%d %H:%M'),
            'mode': _title,
            'total': total,
            'correct': correct_count,
            'score': score,
            'wrong_topics': list(set(q['_topic'] for q in wrong_list)),
        }
        records = load_records()
        records['history'].append(test_record)

        for q in wrong_list:
            wrong_record = {
                'id': q['id'],
                'topic': q['_topic'],
                'question': q['question'],
                'correct_answer': q['answer'],
                'explanation': q['explanation'],
                'date': datetime.now().strftime('%Y-%m-%d %H:%M'),
            }
            if not any(w['id'] == q['id'] for w in records['wrong_questions']):
                records['wrong_questions'].append(wrong_record)

        save_records(records)
        print('\n测验记录已保存！')

✏️ 请在 my_answers 列表中填写 5 个答案：
   my_answers[0] = ""  # 第1题
   my_answers[1] = ""  # 第2题
   my_answers[2] = ""  # 第3题
   my_answers[3] = ""  # 第4题
   my_answers[4] = ""  # 第5题

填写完毕后，再次运行此单元格进行批改。


---
## 📕 错题本

运行下方代码查看历史错题：

In [ ]:
records = load_records()
wrong_qs = records.get('wrong_questions', [])

WRONG_STYLE = """
<style>
.wb-banner {
    background: linear-gradient(90deg, #e53e3e 0%, #c53030 100%);
    color: white; padding: 12px 20px; border-radius: 8px;
    margin: 8px 0 16px 0; font-size: 18px; font-weight: bold;
    display: flex; justify-content: space-between; align-items: center;
}
.wb-empty {
    text-align: center; padding: 40px; color: #48bb78;
    font-size: 18px; font-weight: bold;
}
.wb-topic {
    background: #fff5f5; border: 1px solid #fed7d7;
    border-radius: 8px; padding: 4px 14px; margin: 18px 0 6px 0;
    font-weight: bold; color: #9b2c2c; font-size: 15px;
    display: inline-block;
}
.wb-card {
    border: 1px solid #e2e8f0; border-left: 4px solid #e53e3e;
    border-radius: 0 10px 10px 0; padding: 16px 22px; margin: 10px 0;
    background: #fff; box-shadow: 0 1px 4px rgba(0,0,0,0.06);
}
.wb-q { font-size: 14px; line-height: 1.8; color: #2d3748; margin-bottom: 10px; }
.wb-ans { display: inline-block; background: #c6f6d5; padding: 4px 12px;
    border-radius: 6px; font-weight: bold; color: #22543d; font-size: 14px; }
.wb-exp { font-size: 13px; color: #718096; margin-top: 8px; line-height: 1.7; }
.wb-date { font-size: 12px; color: #a0aec0; text-align: right; margin-top: 6px; }
</style>
"""

if not wrong_qs:
    display(HTML(WRONG_STYLE))
    display(HTML('<div class="wb-empty">🎉 错题本为空！你还没有做错过题目，继续保持！</div>'))
else:
    display(HTML(WRONG_STYLE))
    display(HTML(f"""
    <div class="wb-banner">
      <span>📕 错题本</span>
      <span style="font-size:14px;opacity:0.9;">共 {len(wrong_qs)} 题</span>
    </div>"""))

    from collections import defaultdict
    grouped = defaultdict(list)
    for q in wrong_qs:
        grouped[q['topic']].append(q)

    html = ''
    for topic, questions in grouped.items():
        html += f'<div class="wb-topic">📌 {topic}（{len(questions)}题）</div>'
        for q in questions:
            exp = str(q.get('explanation', '待补充')).replace('\n', '<br>')
            html += f"""
            <div class="wb-card">
              <div class="wb-q"><b>题目 {q['id']}：</b>{q['question'][:200]}</div>
              <div><span class="wb-ans">正确答案：{q['correct_answer']}</span></div>
              <div class="wb-exp">解析：{exp}</div>
              <div class="wb-date">记录时间：{q['date']}</div>
            </div>"""
    display(HTML(html))

### 🎉 错题本为空！你还没有做错过题目，继续保持！

---
## 📈 学情分析

运行下方代码查看学习进度和薄弱环节分析：

In [ ]:
records = load_records()
history = records.get('history', [])

STAT_STYLE = """
<style>
.stat-banner {
    background: linear-gradient(90deg, #667eea 0%, #764ba2 100%);
    color: white; padding: 14px 24px; border-radius: 10px;
    margin: 8px 0 16px 0; font-size: 20px; font-weight: bold;
}
.stat-grid {
    display: grid; grid-template-columns: repeat(auto-fit, minmax(140px,1fr));
    gap: 12px; margin: 16px 0;
}
.stat-item {
    background: #f7fafc; border: 1px solid #e2e8f0; border-radius: 10px;
    padding: 16px; text-align: center; box-shadow: 0 1px 3px rgba(0,0,0,0.06);
}
.stat-val { font-size: 28px; font-weight: bold; color: #2d3748; }
.stat-lbl { font-size: 13px; color: #718096; margin-top: 4px; }
.weak-table {
    width: 100%; border-collapse: separate; border-spacing: 0;
    border-radius: 8px; overflow: hidden; margin: 12px 0;
    box-shadow: 0 1px 4px rgba(0,0,0,0.08);
}
.weak-table th {
    background: #edf2f7; color: #4a5568; padding: 8px 14px;
    font-size: 13px; text-align: center; border-bottom: 2px solid #cbd5e0;
}
.weak-table td {
    padding: 7px 14px; font-size: 14px; border-bottom: 1px solid #edf2f7;
    text-align: center;
}
.hist-table { width: 100%; border-collapse: separate; border-spacing: 0;
    border-radius: 8px; overflow: hidden; margin: 12px 0;
    box-shadow: 0 1px 4px rgba(0,0,0,0.08); }
.hist-table th { background: #edf2f7; color: #4a5568; padding: 8px 14px;
    font-size: 13px; border-bottom: 2px solid #cbd5e0; }
.hist-table td { padding: 7px 14px; font-size: 14px;
    border-bottom: 1px solid #edf2f7; text-align: center; }
.trend-box { background: #f0fff4; border: 1px solid #c6f6d5; border-radius: 8px;
    padding: 14px 20px; margin: 16px 0; font-size: 16px; font-weight: bold; text-align: center; }
.trend-down { background: #fff5f5; border-color: #fed7d7; }
.trend-flat { background: #fffff0; border-color: #fefcbf; }
</style>
"""

if not history:
    display(HTML('<div style="text-align:center;padding:40px;color:#718096;font-size:16px;">还没有测验记录，完成一次测验后再来查看！</div>'))
else:
    display(HTML(STAT_STYLE))
    display(HTML('<div class="stat-banner">📈 学情分析报告</div>'))

    total_tests = len(history)
    total_questions = sum(h['total'] for h in history)
    total_correct = sum(h['correct'] for h in history)
    avg_score = round(sum(h['score'] for h in history) / total_tests)
    rate = round(total_correct / total_questions * 100)

    display(HTML(f"""
    <div class="stat-grid">
      <div class="stat-item"><div class="stat-val">{total_tests}</div><div class="stat-lbl">测验次数</div></div>
      <div class="stat-item"><div class="stat-val">{total_questions}</div><div class="stat-lbl">总做题数</div></div>
      <div class="stat-item"><div class="stat-val">{rate}%</div><div class="stat-lbl">总正确率</div></div>
      <div class="stat-item"><div class="stat-val">{avg_score}</div><div class="stat-lbl">平均分</div></div>
    </div>"""))

    # 薄弱知识点
    from collections import Counter
    wrong_topic_counter = Counter()
    for h in history:
        for t in h.get('wrong_topics', []):
            wrong_topic_counter[t] += 1

    if wrong_topic_counter:
        rows = ''
        for rank, (topic, count) in enumerate(wrong_topic_counter.most_common(), 1):
            if count >= 3:
                badge = '<span style="color:#e53e3e;font-weight:bold;">🔴 急需复习</span>'
            elif count >= 2:
                badge = '<span style="color:#d69e2e;font-weight:bold;">🟡 建议复习</span>'
            else:
                badge = '<span style="color:#38a169;">🟢 偶尔出错</span>'
            rows += f'<tr><td>{rank}</td><td style="text-align:left;">{topic}</td><td>{count}</td><td>{badge}</td></tr>'

        display(HTML(f"""
        <h3 style="color:#4a5568;margin:16px 0 8px;">⚠️ 薄弱知识点排行</h3>
        <table class="weak-table">
          <thead><tr><th style="width:50px">排名</th><th>知识点</th><th style="width:80px">出错次数</th><th style="width:100px">建议</th></tr></thead>
          <tbody>{rows}</tbody>
        </table>"""))

    # 历次记录
    hist_rows = ''
    for h in reversed(history[-10:]):
        s = h['score']
        sc = '#38a169' if s >= 80 else ('#d69e2e' if s >= 60 else '#e53e3e')
        hist_rows += f'<tr><td>{h["date"]}</td><td style="text-align:left;">{h["mode"]}</td><td>{h["correct"]}/{h["total"]}</td><td style="color:{sc};font-weight:bold;">{s}分</td></tr>'

    display(HTML(f"""
    <h3 style="color:#4a5568;margin:16px 0 8px;">📅 历次测验记录</h3>
    <table class="hist-table">
      <thead><tr><th>日期</th><th>模式</th><th>正确/总题</th><th>得分</th></tr></thead>
      <tbody>{hist_rows}</tbody>
    </table>"""))

    # 进步趋势
    if len(history) >= 3:
        recent_3 = [h['score'] for h in history[-3:]]
        earlier_3 = [h['score'] for h in history[:3]]
        trend = sum(recent_3)/3 - sum(earlier_3)/3
        if trend > 5:
            display(HTML(f'<div class="trend-box">📈 进步趋势：+{trend:.0f}分，你在进步！继续保持！</div>'))
        elif trend < -5:
            display(HTML(f'<div class="trend-box trend-down">📉 成绩有所下降（{trend:.0f}分），建议回顾薄弱知识点。</div>'))
        else:
            display(HTML(f'<div class="trend-box trend-flat">➡️ 成绩稳定，继续努力向满分冲刺！</div>'))

### ⚠️ 还没有测验记录，完成一次测验后再来查看！

---
## 🔄 清空错题本

如果你觉得已经掌握了所有错题，可以清空错题本重新开始：

In [22]:
# ⚠️ 取消注释下面两行来清空错题本（谨慎操作！）
# records['wrong_questions'] = []
# save_records(records)
# print('🗑️ 错题本已清空！')

print('如需清空错题本，请取消上方代码的注释后运行。')

如需清空错题本，请取消上方代码的注释后运行。
